# kashi — CTC bootstrap (Colab GPU)

Fine-tunes a Japanese SSL encoder with CTC on the karaoke label **sequences**
(timestamps unused → jitter-immune), then **forced-aligns** every song to produce
sharp per-frame labels for the local textless decoder (SIGNOFFS S9/S10).

**Colab:** upload `artifacts/colab_pack.zip` to Drive as
`MyDrive/Projects/krke_ctn/kashi/colab_pack.zip`; GPU runtime.
**Kaggle:** upload `colab_pack.zip` as a (private) Kaggle Dataset and attach it
via *+ Add Input*; Settings: Accelerator **GPU**, Internet **On**, Persistence **Files only**.
Outputs land in `/kaggle/working/kashi/`.

**Intermediates persist on Drive** under `.../kashi/out/` — the REUSE flags below
skip finished stages on re-runs (set one to `False` to force recompute).

**After:** download `MyDrive/Projects/krke_ctn/kashi/ctc_out.zip`, then locally:
`kashi colab integrate ~/Downloads/ctc_out.zip`.

In [ ]:
# ==== run flags: reuse saved intermediates? (False = recompute that stage) ====
REUSE_DATA  = True    # extracted colab_pack under <DRIVE>/data
REUSE_MODEL = True    # trained CTC model at <DRIVE>/out/ctc_model
REUSE_FA    = True    # per-song forced-alignment outputs at <DRIVE>/out/fa_*

EPOCHS          = 12
BATCH           = 4     # ~15 s crops; raise on A100
GRAD_ACCUM      = 2
DEBUG_MAX_SONGS = None  # e.g. 2 = tiny smoke run (2 train / 1 test songs)
PSEUDO_DIR      = ''    # P5: dir with manifest.jsonl + crops/*.npy (spike-decoded
                        # pseudo-transcripts from `kashi loop ctc-harvest`); '' = labeled only
PSEUDO_MIN_CONF = 0.5   # keep pseudo crops with mean spike confidence >= this
PSEUDO_WEIGHT   = 0.3   # CTC loss weight for pseudo crops (labeled crops = 1.0)


In [ ]:
%%capture
!pip install -q "transformers>=4.44" soundfile

In [ ]:
import os, glob, zipfile, torch, torchaudio, transformers

IN_KAGGLE = os.path.exists('/kaggle')
DATA = None
if IN_KAGGLE:
    # outputs -> /kaggle/working (Settings -> Persistence: "Files only")
    DRIVE = '/kaggle/working/kashi'
    os.makedirs(DRIVE, exist_ok=True)
    # input dataset: either the raw zip or an already-extracted pack
    hits = glob.glob('/kaggle/input/**/tokens.txt', recursive=True)
    if hits:
        DATA = os.path.dirname(hits[0])          # read-only extracted pack
    else:
        packs = glob.glob('/kaggle/input/**/colab_pack.zip', recursive=True)
        assert packs, 'Attach a Dataset with colab_pack.zip or the extracted pack (+ Add Input)'
        PACK = packs[0]
    try:  # optional HF token from Kaggle secrets (public models work without it)
        from kaggle_secrets import UserSecretsClient
        os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
        print('HF_TOKEN loaded from Kaggle secrets')
    except Exception:
        pass
else:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception:
        pass  # local testing: set KASHI_DRIVE
    DRIVE = os.environ.get('KASHI_DRIVE', '/content/drive/MyDrive/Projects/krke_ctn/kashi')
    PACK = f'{DRIVE}/colab_pack.zip'

if DATA is None:
    DATA = f'{DRIVE}/data'
    if not (REUSE_DATA and os.path.exists(f'{DATA}/tokens.txt')):
        os.makedirs(DATA, exist_ok=True)
        with zipfile.ZipFile(PACK) as z:
            z.extractall(DATA)
OUT = f'{DRIVE}/out'
MODEL_DIR = f'{OUT}/ctc_model'
for d in (OUT, f'{OUT}/fa_labels', f'{OUT}/fa_segments'):
    os.makedirs(d, exist_ok=True)
print(torch.__version__, transformers.__version__, 'cuda:', torch.cuda.is_available())
print('DATA:', DATA)
print('DATA contents:', sorted(os.listdir(DATA)))

In [ ]:
# ---- config ----
CHECKPOINT = 'yky-h/japanese-hubert-base'   # verified public MIRROR of rinna/japanese-hubert-base
# Alternatives (public, verified to resolve):
# CHECKPOINT = 'prj-beatrice/japanese-hubert-base-phoneme-ctc'  # same rinna model, already
#   CTC-tuned on JA phonemes — warm start; head re-initialised for our 110 morae
# CHECKPOINT = 'reazon-research/japanese-wav2vec2-base-rs35kh'  # wav2vec2, 35k h ReazonSpeech v2
LR_ENCODER  = 3e-5
LR_HEAD     = 1e-3
CROP_S      = 15.0
SR          = 16000
FRAME_MS    = 20

In [ ]:
# ---- tokens & splits (index 109 = <silence> = CTC blank) ----
TOKENS = open(f'{DATA}/tokens.txt').read().splitlines()
assert len(TOKENS) == 110 and TOKENS[-1] == '<silence>'
TOK2ID = {t: i for i, t in enumerate(TOKENS)}
BLANK = 109
split = dict(l.split(':') for l in open(f'{DATA}/split.txt').read().splitlines())
TRAIN_IDS = [int(x) for x in split['train'].split(',')]
TEST_IDS  = [int(x) for x in split['test'].split(',')]
if DEBUG_MAX_SONGS:
    TRAIN_IDS, TEST_IDS = TRAIN_IDS[:DEBUG_MAX_SONGS], TEST_IDS[:1]
print(len(TRAIN_IDS), 'train /', len(TEST_IDS), 'test songs')

In [ ]:
# ---- crops: cut songs at silences/exclusions into <=CROP_S chunks with token sequences ----
import csv, soundfile as sf, numpy as np

def read_rows(song_id):
    with open(f'{DATA}/subtitles/{song_id}.csv', newline='') as f:
        return [dict(r) for r in csv.DictReader(f)]

def song_crops(song_id):
    rows = read_rows(song_id)
    crops, cur, t0, last_e = [], [], None, 0.0
    def flush(t_end):
        nonlocal cur, t0
        if cur and t_end - t0 >= 1.0:
            crops.append((t0, t_end, [TOK2ID[t] for t in cur]))
        cur, t0 = [], None
    for r in rows:
        tok = r['token']; s, e = float(r['start']), float(r['end'])
        excl = r.get('exclude', 'False').strip().lower() == 'true'
        if tok == '<silence>' or excl or tok not in TOK2ID:
            if cur and (excl or (e - s) > 0.6):
                flush(s)
            continue
        if t0 is None: t0 = max(0.0, s - 0.1)
        if e - t0 > CROP_S:
            flush(s)
            t0 = max(0.0, s - 0.1)
        cur.append(tok)
        last_e = e
    if cur: flush(last_e + 0.1)
    return crops

CROPS = {sid: song_crops(sid) for sid in TRAIN_IDS + TEST_IDS}
print('crops:', sum(len(v) for v in CROPS.values()))

In [ ]:
# ---- preload audio to RAM (mp3 has no sample-accurate seek: per-crop reads
# would re-decode from the file start — catastrophic on FUSE mounts) ----
import time
from torch.utils.data import Dataset, DataLoader

def load_song(sid):
    wave, srr = sf.read(f'{DATA}/vocals/{sid}.mp3', dtype='float32', always_2d=True)
    wave = wave.mean(1)
    if srr != SR:
        wave = torchaudio.functional.resample(torch.from_numpy(wave), srr, SR).numpy()
    return wave.astype(np.float16)   # ~750 MB for all 93 songs

AUDIO = {}
t0 = time.time()
for i, sid in enumerate(TRAIN_IDS + TEST_IDS, 1):
    AUDIO[sid] = load_song(sid)
    if i % 10 == 0 or i == len(TRAIN_IDS) + len(TEST_IDS):
        print(f'preloaded {i}/{len(TRAIN_IDS)+len(TEST_IDS)} songs '
              f'({time.time()-t0:.0f}s)', flush=True)

PSEUDO = []
if PSEUDO_DIR:  # P5: pseudo-transcript crops mixed into training (never into test)
    import json as _json
    with open(f'{PSEUDO_DIR}/manifest.jsonl') as f:
        _rows = [_json.loads(l) for l in f]
    PSEUDO = [m for m in _rows if m['conf_mean'] >= PSEUDO_MIN_CONF]
    print(f'pseudo crops: {len(PSEUDO)}/{len(_rows)} pass conf>={PSEUDO_MIN_CONF} '
          f'({sum(m["dur_s"] for m in PSEUDO)/3600:.2f} h, loss weight {PSEUDO_WEIGHT})')

class CropSet(Dataset):
    def __init__(self, ids, pseudo=()):
        self.items = [(sid, c, 1.0) for sid in ids for c in CROPS[sid]]
        self.items += [(None, m, PSEUDO_WEIGHT) for m in pseudo]
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        sid, c, w = self.items[i]
        if sid is None:   # pseudo crop: fp16 audio slice on disk
            wave = np.load(f'{PSEUDO_DIR}/{c["file"]}').astype(np.float32)
            labels = c['tokens']
        else:
            t0, t1, labels = c
            wave = AUDIO[sid][int(t0*SR):int(t1*SR)].astype(np.float32)
        wave = (wave - wave.mean()) / (wave.std() + 1e-7)
        return torch.from_numpy(wave), torch.tensor(labels), w

def collate(batch):
    waves, labels, ws = zip(*batch)
    wl = torch.tensor([len(w) for w in waves])
    ll = torch.tensor([len(l) for l in labels])
    wpad = torch.zeros(len(waves), int(wl.max()))
    for i, w in enumerate(waves): wpad[i, :len(w)] = w
    return wpad, wl, torch.cat(labels), ll, torch.tensor(ws)

train_dl = DataLoader(CropSet(TRAIN_IDS, PSEUDO), batch_size=BATCH, shuffle=True,  collate_fn=collate, num_workers=0)
test_dl  = DataLoader(CropSet(TEST_IDS),          batch_size=BATCH, shuffle=False, collate_fn=collate, num_workers=0)

In [ ]:
# ---- model: load finished one from Drive, else build from CHECKPOINT ----
from transformers import AutoModelForCTC
NEED_TRAIN = not (REUSE_MODEL and os.path.exists(f'{MODEL_DIR}/config.json'))
src = CHECKPOINT if NEED_TRAIN else MODEL_DIR
model = AutoModelForCTC.from_pretrained(
    src, vocab_size=110, pad_token_id=BLANK, ctc_loss_reduction='mean',
    ignore_mismatched_sizes=True).cuda()
model.freeze_feature_encoder()
print('training needed:' if NEED_TRAIN else 'reusing trained model:', src)

In [ ]:
# ---- SER (edit distance) for eval ----
def lev(a, b):
    if not a: return len(b)
    prev = list(range(len(b)+1))
    for i, x in enumerate(a, 1):
        cur = [i] + [0]*len(b)
        for j, y in enumerate(b, 1):
            cur[j] = min(prev[j]+1, cur[j-1]+1, prev[j-1] + (x != y))
        prev = cur
    return prev[-1]

@torch.no_grad()
def evaluate():
    model.eval(); d = n = 0
    for wpad, wl, labels, ll, _w in test_dl:
        with torch.autocast('cuda', torch.float16):
            logits = model(wpad.cuda()).logits
        preds = logits.argmax(-1).cpu()
        off = 0
        for i in range(len(wl)):
            T_i = model._get_feat_extract_output_lengths(wl[i]).item()
            p = preds[i, :T_i]
            hyp = [int(x) for x in torch.unique_consecutive(p) if int(x) != BLANK]
            ref = labels[off:off+ll[i]].tolist(); off += ll[i]
            d += lev(hyp, ref); n += len(ref)
    model.train(); return d / max(1, n)

In [ ]:
# ---- train (skipped when reusing) ----
if NEED_TRAIN:
    head = [p for nm, p in model.named_parameters() if 'lm_head' in nm]
    rest = [p for nm, p in model.named_parameters() if 'lm_head' not in nm and p.requires_grad]
    opt = torch.optim.AdamW([{'params': rest, 'lr': LR_ENCODER}, {'params': head, 'lr': LR_HEAD}])
    scaler = torch.amp.GradScaler('cuda')
    ctc = torch.nn.CTCLoss(blank=BLANK, zero_infinity=True, reduction='none')
    best = 9e9
    for epoch in range(1, EPOCHS+1):
        tot = steps = 0
        for k, (wpad, wl, labels, ll, w) in enumerate(train_dl):
            with torch.autocast('cuda', torch.float16):
                logits = model(wpad.cuda()).logits          # [B, T, 110]
            logp = torch.log_softmax(logits.float(), -1).transpose(0, 1)
            out_len = model._get_feat_extract_output_lengths(wl).cuda()
            raw = ctc(logp, labels.cuda(), out_len, ll.cuda())      # [B] NLL
            # == reduction='mean' when all weights are 1 (labeled-only runs)
            loss = ((raw / ll.cuda().clamp(min=1)) * w.cuda()).mean() / GRAD_ACCUM
            scaler.scale(loss).backward()
            if (k+1) % GRAD_ACCUM == 0:
                scaler.step(opt); scaler.update(); opt.zero_grad()
            tot += float(loss.detach()) * GRAD_ACCUM; steps += 1
            if steps % 100 == 0:
                import time as _t
                print(f'  epoch {epoch} step {steps}/{len(train_dl)} loss={tot/steps:.3f}', flush=True)
        ser = evaluate()
        print(f'epoch {epoch}/{EPOCHS} loss={tot/max(1,steps):.4f} test_SER={ser:.4f}', flush=True)
        if ser < best:
            best = ser
            model.save_pretrained(MODEL_DIR)   # persists on Drive immediately
    print('best test SER (greedy, no LM):', best)
    model = AutoModelForCTC.from_pretrained(MODEL_DIR).cuda()  # reload best
    model.eval()
else:
    best = evaluate()
    print('skipped (REUSE_MODEL); greedy test SER of saved model:', best)

In [ ]:
# ---- forced alignment of EVERY labeled song (S9: training bootstrap only) ----
import numpy as np
model.eval()
# guard: a blank-collapsed model (SER ~1.0) produces garbage alignments
assert best < 0.95, (
    f'model greedy SER={best:.3f} — refusing to force-align with a blank-collapsed '
    'model. Retrain (REUSE_MODEL=False) or investigate before FA.')

@torch.no_grad()
def song_emissions(sid, chunk_s=30.0, ov_s=2.0):
    wave = AUDIO[sid].astype(np.float32)
    wave = (wave - wave.mean()) / (wave.std() + 1e-7)
    hop = SR * FRAME_MS // 1000
    parts, step = [], int((chunk_s - ov_s) * SR)
    for s in range(0, len(wave), step):
        piece = torch.from_numpy(wave[max(0, s-int(ov_s*SR)) : s+int(chunk_s*SR)].copy()).float()
        with torch.autocast('cuda', torch.float16):
            lg = model(piece[None].cuda()).logits[0].float().cpu()
        lead = (s - max(0, s-int(ov_s*SR))) // hop
        keep = min(int((chunk_s-ov_s)*SR)//hop, len(lg)-lead)
        parts.append(lg[lead:lead+keep])
        if s + int(chunk_s*SR) >= len(wave): break
    em = torch.cat(parts)
    return torch.log_softmax(em, -1)

for sid in TRAIN_IDS + TEST_IDS:
    if REUSE_FA and os.path.exists(f'{OUT}/fa_labels/{sid}.npy'):
        continue
    rows = read_rows(sid)
    seq = [TOK2ID[r['token']] for r in rows
           if r['token'] in TOK2ID and r['token'] != '<silence>'
           and r.get('exclude','False').strip().lower() != 'true']
    if not seq: continue
    em = song_emissions(sid)
    try:
        path, scores = torchaudio.functional.forced_align(
            em[None].contiguous(), torch.tensor([seq], dtype=torch.int32), blank=BLANK)
    except Exception as e:
        print(sid, 'FA failed:', e); continue
    frame = path[0].numpy().astype(np.int16)   # class per frame; blank=109=<silence>
    Tf = len(frame)                            # == emission length (may be < len(wave)//hop)
    np.save(f'{OUT}/fa_labels/{sid}.npy', frame)
    with open(f'{OUT}/fa_segments/{sid}.csv', 'w') as f:
        f.write('start,end,token,conf\n')
        t = 0
        sc = scores[0].exp().numpy()
        while t < Tf:
            u = t
            while u < Tf and frame[u] == frame[t]: u += 1
            if frame[t] != BLANK:
                f.write(f'{t*0.02:.3f},{u*0.02:.3f},{TOKENS[frame[t]]},{sc[t:u].mean():.3f}\n')
            t = u
    print(sid, 'ok', flush=True)
print('FA outputs:', len(os.listdir(f'{OUT}/fa_labels')), 'songs')

In [ ]:
# ---- package results (zip of the Drive out/ dir) ----
import json, shutil
json.dump({'checkpoint': CHECKPOINT, 'epochs': EPOCHS, 'greedy_test_ser': best,
           'pseudo_crops': len(PSEUDO), 'pseudo_min_conf': PSEUDO_MIN_CONF if PSEUDO_DIR else None,
           'pseudo_weight': PSEUDO_WEIGHT if PSEUDO_DIR else None,
           'fa_songs': len(os.listdir(f'{OUT}/fa_labels'))},
          open(f'{OUT}/metrics.json', 'w'))
shutil.make_archive(f'{DRIVE}/ctc_out', 'zip', OUT)
where = 'the notebook Output pane (or Save Version -> Output)' if IN_KAGGLE else 'Drive'
print(f'wrote {DRIVE}/ctc_out.zip — download it from {where}, then locally: kashi colab integrate ctc_out.zip')